# Week 1 — Python and image-array foundations

This lab uses the shared `uv` environment. Run JupyterLab from the course root.

## Use of generative AI

Generative AI was used to assist the preparation of these teaching materials. The teaching team is responsible for their content.


## Example outputs and independent implementation

The supplied reference code keeps the notebook runnable and shows example outputs for visual comparison. You may use the examples and hints to understand the task and plan an approach, but write your own implementation. Do not copy, adapt, translate, or import the supplied reference implementations into your assessed code.

Where function-implementation tasks are provided, student cells are labelled **Your implementation** and start with an unfinished-task marker. A separate preview runner displays example results until you replace that marker; it prints **REFERENCE PREVIEW — TASK INCOMPLETE**. Passing checks on a reference preview does not complete the task. Demonstration code is collapsed by default; expand a conceptual hint, where provided, if you are stuck. Once your code is written, its errors are shown normally. Use `preview.use_reference = False` to check that implemented tasks run without reference previews, where a preview session is provided.

Explain your reasoning and verify your results. The supplied Python files remain inspectable; the preview runner is a learning aid, not a plagiarism detector.


In [ ]:
from pathlib import Path
from time import perf_counter

import cv2
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open this notebook inside the COMP3419 repository"
IMAGE_PATH = ROOT / "COMP3419-W01-Lab-Files" / "sample_image.jpg"
image = imageio.imread(IMAGE_PATH)
print(f"shape={image.shape}, dtype={image.dtype}, range=[{image.min()}, {image.max()}]")

## 1. Shape, coordinates, and channels

A colour image array normally has shape `(height, width, channels)`. NumPy indexes it as `[row, column, channel]`, while plotting and pointer APIs usually describe positions as `(x, y) = (column, row)`.

In [ ]:
height, width, channels = image.shape
x, y = width // 3, height // 2
rgb_at_xy = image[y, x]
print(f"pixel at display coordinate ({x}, {y}) = RGB {rgb_at_xy}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image)
axes[0].scatter([x], [y], c="white", edgecolors="black")
axes[0].set_title("RGB image and (x, y) point")
axes[1].imshow(image[..., 0], cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Red channel")
for axis in axes:
    axis.axis("off")
plt.tight_layout()

## 2. Slices, views, and copies

Many NumPy slices are views: modifying them can modify the original array. Use `.copy()` when an independent image is required.

In [ ]:
row_slice = slice(height // 4, 3 * height // 4)
col_slice = slice(width // 4, 3 * width // 4)
crop = image[row_slice, col_slice].copy()
assert not np.shares_memory(crop, image)

rgb_reordered = image[..., [2, 1, 0]]
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(crop)
axes[0].set_title(f"crop: {crop.shape}")
axes[1].imshow(rgb_reordered)
axes[1].set_title("R/B channels exchanged")
for axis in axes:
    axis.axis("off")
plt.tight_layout()

## 3. Dtype and overflow

`uint8` stores integers from 0 to 255. Convert to a wider signed integer or floating-point dtype before arithmetic, then clip to the valid range and convert back.

In [ ]:
unsafe = image + np.uint8(80)
safe = np.clip(image.astype(np.int16) + 80, 0, 255).astype(np.uint8)

print("bright pixels before:", image.max())
print("unsafe result max:", unsafe.max(), "(wrapped values are possible)")
print("safe result max:", safe.max())
assert safe.dtype == np.uint8 and safe.min() >= 0 and safe.max() <= 255

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, data, title in zip(axes, [image, unsafe, safe], ["original", "unsafe uint8", "safe"]):
    axis.imshow(data)
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()

## 4. Vectorisation and validation

The mask below selects red-dominant pixels in one array expression. Timing is illustrative; repeat measurements before drawing performance conclusions.

In [ ]:
start = perf_counter()
red_mask = (image[..., 0].astype(np.int16) - image[..., 1].astype(np.int16) > 35) & (image[..., 0] > 120)
masked = np.where(red_mask[..., None], image, 0)
elapsed_ms = (perf_counter() - start) * 1000

assert red_mask.shape == image.shape[:2]
assert red_mask.dtype == np.bool_
print(f"selected {red_mask.mean():.1%} of pixels in {elapsed_ms:.3f} ms")
plt.figure(figsize=(7, 4))
plt.imshow(masked)
plt.axis("off")
plt.title("Vectorised red-dominance mask");

## 5. RGB versus BGR

ImageIO returns RGB. OpenCV's default colour image loader returns BGR. Convert explicitly at library boundaries.

In [ ]:
bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
rgb_from_cv = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
mean_absolute_decode_difference = np.abs(rgb_from_cv.astype(np.int16) - image.astype(np.int16)).mean()
print(f"mean absolute decoder difference: {mean_absolute_decode_difference:.3f}")
assert rgb_from_cv.shape == image.shape
# JPEG decoders can differ slightly, so exact equality is not required.
assert mean_absolute_decode_difference < 2

## Optional orientation self-check

This formative exercise is not submitted. Complete the cell below without a pixel loop, then explain one coordinate, channel-order, view/copy, or dtype bug that your assertions prevent.

In [ ]:
# Inputs: image is an RGB uint8 array with shape (H, W, 3) and values in [0, 255].
# Expected output: orientation_crop is an independent uint8 array with shape
# (H//2, W//2, 3); orientation_bright is a same-shape uint8 result using
# saturated addition by 40; orientation_mask is a Boolean (H, W) array that is
# True exactly where the blue channel exceeds both red and green.

# TODO 1: extract the central half of the image as an independent copy.
orientation_crop = None

# TODO 2: increase brightness by 40 without uint8 overflow.
orientation_bright = None

# TODO 3: create a Boolean mask for pixels whose blue channel exceeds both red and green.
orientation_mask = None

# Uncomment after completing the tasks.
# assert orientation_crop.shape[:2] == (height // 2, width // 2)
# assert not np.shares_memory(orientation_crop, image)
# assert orientation_bright.dtype == np.uint8
# assert orientation_mask.shape == image.shape[:2] and orientation_mask.dtype == np.bool_